# New THINGS head (1306 non-overlapping concepts)

Linear(512, 1306) on the frozen pretrained CordsNet-R8 features (classifier input, steps 70-99); CordsNet's weighted CE, true THINGS labels; AdamW, weight decay on the weights only; lr 1e-3 cosine to 0, 40 epochs.
Train 13,060 images (10 per concept), val 1,306 (1 per concept). Two selections over weight decay x epoch: **top1** = best val top-1 at the last step (200 ms), **ce** = lowest val CE (steps 70-99).
Chance: top-1 1/1306 = 0.08%, top-5 0.38%.

Input: `Round3/outputs/things_head/cosine_ep40/` (`history.csv`, `results_top1.npz`, `results_ce.npz`, `config.json`; from `Round3/things_head/train_head.py`, copied from Torch).

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

CN = Path("~/Desktop/DynVision/CordsNet").expanduser()
out = CN / "Round3/outputs/things_head/cosine_ep40"
h = pd.read_csv(out / "history.csv")
res = {sel: np.load(out / f"results_{sel}.npz") for sel in ["top1", "ce"]}
cfg = json.load(open(out / "config.json"))
wds = sorted(h["weight_decay"].unique())
colors = ["#2a78d6", "#eb6834", "#1a9e77", "#8e44ad", "#c0392b"]
color = {wd: colors[i] for i, wd in enumerate(wds)}
marker = {"top1": "o", "ce": "s"}
print(cfg)
for sel, r in res.items():
    print(f"selected by {sel}: weight decay {float(r['weight_decay']):g}, epoch {int(r['epoch'])}")

## 1. Per weight decay

For each selection: the best epoch and its val top-1 / top-5 / CE, train top-1. Last column: val top-1 at the last epoch.

In [ ]:
rows = []
for wd in wds:
    g = h[h["weight_decay"] == wd]
    bt, bc = g.loc[g["val_top1"].idxmax()], g.loc[g["val_ce"].idxmin()]
    rows.append({"weight decay": wd,
                 "top1: epoch": int(bt["epoch"]), "top1: val top-1": bt["val_top1"], "top1: val top-5": bt["val_top5"], "top1: val CE": bt["val_ce"],
                 "ce: epoch": int(bc["epoch"]), "ce: val top-1": bc["val_top1"], "ce: val top-5": bc["val_top5"], "ce: val CE": bc["val_ce"],
                 "train top-1, last epoch": g["train_top1"].iloc[-1], "val top-1, last epoch": g["val_top1"].iloc[-1],
                 "|W| per class, last epoch": g["W_norm"].iloc[-1]})
table = pd.DataFrame(rows).set_index("weight decay")
table["selected"] = [" ".join(sel for sel, r in res.items() if float(r["weight_decay"]) == wd) for wd in table.index]
table.round(4).T

## 2. Learning curves

Left: top-1 (solid val, dashed train). Right: val CE. Circle: selected by top1; square: selected by ce.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for wd in wds:
    g = h[h["weight_decay"] == wd]
    axes[0].plot(g["epoch"], g["val_top1"], color=color[wd], lw=2, label=f"wd {wd:g}")
    axes[0].plot(g["epoch"], g["train_top1"], color=color[wd], lw=1.2, ls="--")
    axes[1].plot(g["epoch"], g["val_ce"], color=color[wd], lw=2, label=f"wd {wd:g}")
for sel, r in res.items():
    pt = h[(h["weight_decay"] == float(r["weight_decay"])) & (h["epoch"] == int(r["epoch"]))]
    for ax, col in [(axes[0], "val_top1"), (axes[1], "val_ce")]:
        ax.scatter(pt["epoch"], pt[col], s=70, marker=marker[sel], facecolor="none", edgecolor="k", zorder=5, label=f"selected: {sel}")
axes[0].set(xlabel="epoch", ylabel="top-1 (last step)", title="top-1: val (solid), train (dashed)", ylim=(0, 1))
axes[1].set(xlabel="epoch", ylabel="weighted CE (steps 70-99)", title="val CE")
for ax in axes:
    ax.grid(axis="y", color="0.9")
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

## 3. The two selected heads: accuracy over the CE steps

Solid: val top-1; dotted: val top-5; dashed: train top-1.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sel_color = {"top1": "#2a78d6", "ce": "#eb6834"}
for sel, r in res.items():
    t_ms = (r["steps"] + 1) * 2
    name = f"{sel} (wd {float(r['weight_decay']):g}, epoch {int(r['epoch'])})"
    ax.plot(t_ms, r["val_top1_t"], color=sel_color[sel], lw=2, marker="o", ms=3, label=f"{name}: val top-1")
    ax.plot(t_ms, r["val_top5_t"], color=sel_color[sel], lw=2, ls=":", label=f"{name}: val top-5")
    ax.plot(t_ms, r["train_top1_t"], color=sel_color[sel], lw=1.2, ls="--", label=f"{name}: train top-1")
    print(f"{sel}: last step val top-1 {r['val_top1_t'][-1]:.4f}, top-5 {r['val_top5_t'][-1]:.4f}, val CE {float(r['val_ce']):.3f}; "
          f"train top-1 {r['train_top1_t'][-1]:.4f}")
ax.set(xlabel="time after image onset (ms)", ylabel="accuracy", ylim=(0, 1.02))
ax.grid(axis="y", color="0.9")
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=7, loc="lower right")
plt.show()